# Group 3 — Agentic Tasks & Workflow-Level Memory Evaluation

This notebook benchmarks **2 memory-augmented AI systems** on three agentic task datasets (250 questions each).

---

## Why Agentic Tasks Are Harder

Unlike conversational memory (Group 1) or long-document comprehension (Group 2), **agentic tasks** require a system to:

- **Plan multi-step action sequences** rather than recall stored facts
- **Retrieve and reuse past workflows** (procedural memory) when facing similar tasks
- **Integrate external tools** (web search, browser navigation) with memory-augmented reasoning
- **Generalize** stored workflow patterns to novel task instances

These properties make agentic evaluation a stronger test of memory systems than simple QA.

---

## Systems

| System | Memory Approach | Web Search |
|--------|----------------|------------|
| **AWM** | Workflow memory via text-embedding-ada-002 retrieval | DuckDuckGo |
| **Memento** | Case-Based Reasoning (CBR) with sup-simcse-bert | DuckDuckGo |

## Datasets & Metrics

| Dataset | Task | Metric | Questions |
|---------|------|--------|-----------|
| **Mind2Web** | Web navigation (action sequences) | Avg Action F1 % | 250 |
| **FRAMES** | Multi-hop reasoning | Accuracy % (contains-match) | 250 |
| **MuSiQue** | Compositional QA | Contains-Match % | 250 |

**Mind2Web Action F1** measures token overlap between a predicted action string (e.g., `CLICK [button] -> submit`) and the gold action string — capturing partial credit for correctly identifying the action type or target element.

**Contains-match** checks whether the gold answer string appears as a substring (case-insensitive) in the model's response.

---
## 0 — Setup

Install dependencies, load environment variables, and define shared metric utilities.

In [ ]:
# Install common dependencies (run once)
# %pip install openai python-dotenv huggingface_hub pandas tabulate

In [ ]:
import os
import json
import re
import pathlib
from collections import Counter

import pandas as pd
from dotenv import load_dotenv

# Load API keys from .env file (place OPENAI_API_KEY there)
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
assert OPENAI_API_KEY, "Set OPENAI_API_KEY in your .env file"

# ── Result output directory ──────────────────────────────────────────────────
RESULTS_DIR = pathlib.Path("../results/group3")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Data directory (populated in the next cell) ──────────────────────────────
DATA_DIR = pathlib.Path("../data/group3")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Results will be saved to:", RESULTS_DIR.resolve())

In [ ]:
# ── Shared metric functions ───────────────────────────────────────────────────

def _tokenize(text: str) -> Counter:
    """Lowercase, split on whitespace/punctuation, return token Counter."""
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    return Counter(tokens)


def token_f1(prediction: str, ground_truth: str) -> float:
    """
    Compute token-level F1 between prediction and ground_truth strings.
    Used for Mind2Web action_repr evaluation.
    Returns a float in [0, 1].
    """
    pred_tokens = _tokenize(str(prediction))
    gt_tokens = _tokenize(str(ground_truth))
    common = sum((pred_tokens & gt_tokens).values())
    if common == 0:
        return 0.0
    precision = common / sum(pred_tokens.values())
    recall = common / sum(gt_tokens.values())
    return 2 * precision * recall / (precision + recall)


def contains_match(prediction: str, ground_truth: str) -> bool:
    """
    Return True if ground_truth (lowercased) is a substring of prediction (lowercased).
    Used for FRAMES and MuSiQue evaluation.
    """
    return str(ground_truth).lower() in str(prediction).lower()


def score_mind2web(results: list) -> float:
    """Average action_f1 across all results, returned as a percentage."""
    if not results:
        return 0.0
    avg_f1 = sum(r["action_f1"] for r in results) / len(results)
    return 100 * avg_f1


def score_contains_match(results: list, pred_key: str, gt_key: str) -> float:
    """
    Contains-match accuracy as a percentage.
    pred_key: key for model's predicted answer in each result dict
    gt_key:   key for ground truth answer in each result dict
    """
    if not results:
        return 0.0
    correct = sum(
        1 for r in results
        if contains_match(r.get(pred_key, ""), r.get(gt_key, ""))
    )
    return 100 * correct / len(results)


print("Metric functions defined.")

---
## 1 — Download Datasets

Datasets are hosted on HuggingFace at [`darshan3j/memory-systems-eval-datasets`](https://huggingface.co/datasets/darshan3j/memory-systems-eval-datasets).

| File | Records | Notes |
|------|---------|-------|
| `group3/mind2web_250_filtered.json` | 250 | Web navigation tasks; each has `action_reprs` list |
| `group3/frames_250_filtered.json` | 250 | Multi-hop reasoning; short factual answers |
| `group3/musique_250_filtered.json` | 250 | Compositional QA; answerable subset of MuSiQue validation |

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO = "darshan3j/memory-systems-eval-datasets"
DATASET_FILES = {
    "mind2web": "group3/mind2web_250_filtered.json",
    "frames":   "group3/frames_250_filtered.json",
    "musique":  "group3/musique_250_filtered.json",
}

datasets = {}
for name, hf_path in DATASET_FILES.items():
    local_path = DATA_DIR / f"{name}_250_filtered.json"
    if not local_path.exists():
        print(f"Downloading {name} ...")
        downloaded = hf_hub_download(
            repo_id=HF_REPO,
            filename=hf_path,
            repo_type="dataset",
            local_dir=str(DATA_DIR),
        )
        # hf_hub_download may nest into subdirs; copy to flat location if needed
        import shutil
        if pathlib.Path(downloaded) != local_path:
            shutil.copy(downloaded, local_path)
    with open(local_path) as f:
        datasets[name] = json.load(f)
    print(f"{name}: {len(datasets[name])} records loaded")

# Quick preview of each dataset schema
for name, data in datasets.items():
    sample = data[0]
    print(f"\n{name} sample keys: {list(sample.keys())}")

---
## 2 — AWM (Agent Workflow Memory)

**Paper:** [AWM: Agent Workflow Memory (Wang et al., 2024)](https://arxiv.org/abs/2409.07429)  
**GitHub:** https://github.com/WangXinglin/AWM

### How It Works

AWM learns **reusable workflow memories** from past agent trajectories. When an agent completes a task (successfully or not), it extracts a generalizable workflow description and stores it. For future tasks, it retrieves the most relevant past workflows via **text-embedding-ada-002** semantic search, then conditions its planning on those workflows.

- **Mind2Web:** The agent receives a task description and a list of available web elements (no raw HTML), retrieves relevant workflows, and predicts the next action string (`CLICK`, `TYPE`, `SELECT`, etc.).
- **FRAMES / MuSiQue:** The agent can issue DuckDuckGo web searches, retrieve workflow memories for multi-hop QA patterns, and synthesize an answer.

### Install

```bash
pip install openai duckduckgo-search sentence-transformers
```

Requires Python 3.10+.

In [ ]:
# ── AWM Setup ────────────────────────────────────────────────────────────────
# Add the AWM project root to sys.path if running outside the AWM repo.
# Adjust AWM_ROOT to point to your local clone of https://github.com/WangXinglin/AWM

import sys

AWM_ROOT = pathlib.Path("../../AWM")  # adjust if needed
if AWM_ROOT.exists() and str(AWM_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(AWM_ROOT.resolve()))
    print(f"Added AWM to sys.path: {AWM_ROOT.resolve()}")
else:
    print(f"AWM_ROOT={AWM_ROOT.resolve()} — adjust if import errors occur.")

# AWM result file paths
AWM_MIND2WEB_RESULTS = RESULTS_DIR / "awm_mind2web_250_results.json"
AWM_FRAMES_RESULTS   = RESULTS_DIR / "awm_frames_250_results.json"
AWM_MUSIQUE_RESULTS  = RESULTS_DIR / "awm_musique_250_results.json"

print("AWM result paths set.")

In [ ]:
# ── AWM Evaluation Loop ───────────────────────────────────────────────────────
# This cell runs AWM on all three datasets.
# Results are saved incrementally so the cell can be safely interrupted and resumed.
#
# Resume logic: already-processed items (matched by id / annotation_id) are
# skipped so you don't re-spend API credits.

# ---- Import AWM components (adjust imports to match your AWM clone) ----------
# Example import structure — replace with actual AWM module paths:
#
#   from awm.agent import AWMAgent
#   from awm.memory import WorkflowMemory
#
# Consult the AWM README for the correct entry points.

# ---- Helper: load existing results and build seen-id set --------------------
def load_existing_results(path: pathlib.Path) -> list:
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return []


def save_results(path: pathlib.Path, results: list) -> None:
    with open(path, "w") as f:
        json.dump(results, f, indent=2)


# ===== Mind2Web =====
mind2web_data = datasets["mind2web"]
awm_mw_results = load_existing_results(AWM_MIND2WEB_RESULTS)
seen_mw = {r["annotation_id"] for r in awm_mw_results}

for item in mind2web_data:
    ann_id = item["annotation_id"]
    if ann_id in seen_mw:
        continue  # resume: already processed

    task = item["confirmed_task"]
    gold_actions = item["action_reprs"]   # list of gold action strings

    # ── Replace this block with your AWM agent call ──────────────────────────
    # Example:
    #   predicted = awm_agent.predict_action(task, gold_actions)
    # For now we use a placeholder:
    predicted_action_repr = ""  # TODO: replace with AWM agent output
    # ─────────────────────────────────────────────────────────────────────────

    # Compute F1 against the first gold action (or best-of-N if desired)
    best_f1 = max(token_f1(predicted_action_repr, g) for g in gold_actions) if gold_actions else 0.0

    awm_mw_results.append({
        "annotation_id":         ann_id,
        "confirmed_task":        task,
        "predicted_action_repr": predicted_action_repr,
        "gold_action_reprs":     gold_actions,
        "action_f1":             best_f1,
        "success":               best_f1 == 1.0,
    })
    save_results(AWM_MIND2WEB_RESULTS, awm_mw_results)

print(f"AWM Mind2Web: {len(awm_mw_results)} results saved to {AWM_MIND2WEB_RESULTS}")

# ===== FRAMES =====
frames_data = datasets["frames"]
awm_fr_results = load_existing_results(AWM_FRAMES_RESULTS)
seen_fr = {r["id"] for r in awm_fr_results}

for item in frames_data:
    item_id = item["id"]
    if item_id in seen_fr:
        continue

    question     = item["Prompt"]
    ground_truth = item["Answer"]

    # ── Replace with AWM agent call ───────────────────────────────────────────
    model_answer = ""  # TODO: replace with AWM agent output
    # ─────────────────────────────────────────────────────────────────────────

    awm_fr_results.append({
        "id":           item_id,
        "question":     question,
        "ground_truth": ground_truth,
        "model_answer": model_answer,
        "is_correct":   contains_match(model_answer, ground_truth),
    })
    save_results(AWM_FRAMES_RESULTS, awm_fr_results)

print(f"AWM FRAMES: {len(awm_fr_results)} results saved to {AWM_FRAMES_RESULTS}")

# ===== MuSiQue =====
musique_data = datasets["musique"]
awm_mq_results = load_existing_results(AWM_MUSIQUE_RESULTS)
seen_mq = {r["id"] for r in awm_mq_results}

for item in musique_data:
    item_id = item["id"]
    if item_id in seen_mq:
        continue

    question = item["question"]
    answer   = item["answer"]

    # ── Replace with AWM agent call ───────────────────────────────────────────
    model_answer = ""  # TODO: replace with AWM agent output
    # ─────────────────────────────────────────────────────────────────────────

    awm_mq_results.append({
        "id":           item_id,
        "question":     question,
        "answer":       answer,
        "model_answer": model_answer,
        "is_correct":   contains_match(model_answer, answer),
    })
    save_results(AWM_MUSIQUE_RESULTS, awm_mq_results)

print(f"AWM MuSiQue: {len(awm_mq_results)} results saved to {AWM_MUSIQUE_RESULTS}")

In [ ]:
# ── AWM Scoring ───────────────────────────────────────────────────────────────
# Load saved results and compute final metrics.

awm_mw_results = load_existing_results(AWM_MIND2WEB_RESULTS)
awm_fr_results = load_existing_results(AWM_FRAMES_RESULTS)
awm_mq_results = load_existing_results(AWM_MUSIQUE_RESULTS)

awm_mind2web_score = score_mind2web(awm_mw_results)
awm_frames_score   = score_contains_match(awm_fr_results, "model_answer", "ground_truth")
awm_musique_score  = score_contains_match(awm_mq_results, "model_answer", "answer")

print("=" * 50)
print("AWM Results")
print("=" * 50)
print(f"Mind2Web  Avg Action F1:  {awm_mind2web_score:.1f}%  ({len(awm_mw_results)} questions)")
print(f"FRAMES    Accuracy:       {awm_frames_score:.1f}%  ({len(awm_fr_results)} questions)")
print(f"MuSiQue   Contains-Match: {awm_musique_score:.1f}%  ({len(awm_mq_results)} questions)")

---
## 3 — Memento (AgentFly)

**Paper:** Memento (AgentFly)  
**GitHub:** https://github.com/AgentFly/Memento

### How It Works

Memento uses **Case-Based Reasoning (CBR)** for agent memory. Past task episodes are stored as cases (problem + solution). When a new task arrives, Memento retrieves the most similar past cases using **sup-simcse-bert** embeddings, then adapts the retrieved solution to the current situation.

Like AWM, it uses **DuckDuckGo** for live web search on FRAMES and MuSiQue tasks.

### Architecture Notes

- **MCP Server architecture:** Memento spawns up to 7 subprocesses via the Model Context Protocol. Each tool (web search, memory retrieval, etc.) runs as a separate MCP server.
- **Separate Python 3.11 venv required:** The MCP server dependencies are not compatible with Python 3.10/3.13. Create a dedicated venv before running.
- **Must run from `client/` directory** inside the Memento repo.

### Result File Formats

Memento saves Mind2Web results as JSON, but FRAMES and MuSiQue results as **JSONL** (one JSON object per line). The loading helper below handles both.

In [ ]:
# ── Memento Setup ─────────────────────────────────────────────────────────────
# Memento requires its own Python 3.11 venv. This notebook does NOT import
# Memento directly; instead, you run the Memento evaluation script externally
# and load the result files here for scoring.
#
# To run Memento externally:
#   cd /path/to/Memento/client
#   source ../venv311/bin/activate    # Python 3.11 venv
#   python evaluate.py --dataset mind2web --output ../../results/group3/memento_mind2web_250_results.json
#   python evaluate.py --dataset frames   --output ../../results/group3/memento_frames_250_results.jsonl
#   python evaluate.py --dataset musique  --output ../../results/group3/memento_musique_250_results.jsonl

MEMENTO_MIND2WEB_RESULTS = RESULTS_DIR / "memento_mind2web_250_results.json"
MEMENTO_FRAMES_RESULTS   = RESULTS_DIR / "memento_frames_250_results.jsonl"
MEMENTO_MUSIQUE_RESULTS  = RESULTS_DIR / "memento_musique_250_results.jsonl"


def load_jsonl(path: pathlib.Path) -> list:
    """Load a JSONL file into a list of dicts."""
    if not path.exists():
        return []
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


print("Memento result paths set.")
print(f"  Mind2Web (JSON):  {MEMENTO_MIND2WEB_RESULTS}")
print(f"  FRAMES   (JSONL): {MEMENTO_FRAMES_RESULTS}")
print(f"  MuSiQue  (JSONL): {MEMENTO_MUSIQUE_RESULTS}")

In [ ]:
# ── Memento Evaluation Loop ───────────────────────────────────────────────────
# Because Memento requires a separate Python 3.11 venv and MCP subprocesses,
# this cell shows how you WOULD call it if running inline.
# In practice: run the Memento scripts externally, then jump to the Scoring cell.
#
# If you want to run inline anyway (not recommended — start a subprocess instead):

import subprocess

MEMENTO_ROOT   = pathlib.Path("../../AgentFly/Memento")  # adjust to your clone
MEMENTO_CLIENT = MEMENTO_ROOT / "client"
MEMENTO_PYTHON = MEMENTO_ROOT / "venv311" / "bin" / "python"  # Python 3.11 venv

MEMENTO_DATASET_CONFIGS = [
    {
        "dataset":    "mind2web",
        "input":      str((DATA_DIR / "mind2web_250_filtered.json").resolve()),
        "output":     str(MEMENTO_MIND2WEB_RESULTS.resolve()),
        "output_fmt": "json",
    },
    {
        "dataset":    "frames",
        "input":      str((DATA_DIR / "frames_250_filtered.json").resolve()),
        "output":     str(MEMENTO_FRAMES_RESULTS.resolve()),
        "output_fmt": "jsonl",
    },
    {
        "dataset":    "musique",
        "input":      str((DATA_DIR / "musique_250_filtered.json").resolve()),
        "output":     str(MEMENTO_MUSIQUE_RESULTS.resolve()),
        "output_fmt": "jsonl",
    },
]

RUN_MEMENTO_INLINE = False  # Set to True only if venv311 is configured

if RUN_MEMENTO_INLINE:
    for cfg in MEMENTO_DATASET_CONFIGS:
        print(f"Running Memento on {cfg['dataset']} ...")
        result = subprocess.run(
            [
                str(MEMENTO_PYTHON),
                "evaluate.py",
                "--dataset", cfg["dataset"],
                "--input",   cfg["input"],
                "--output",  cfg["output"],
            ],
            cwd=str(MEMENTO_CLIENT),
            capture_output=True,
            text=True,
        )
        print(result.stdout[-2000:] if result.stdout else "(no stdout)")
        if result.returncode != 0:
            print("STDERR:", result.stderr[-1000:])
        print(f"Done: {cfg['dataset']}")
else:
    print("RUN_MEMENTO_INLINE=False — load pre-computed results in the Scoring cell.")

In [ ]:
# ── Memento Scoring ───────────────────────────────────────────────────────────
# Load saved result files and compute metrics.
# Mind2Web results are JSON; FRAMES and MuSiQue results are JSONL.
# Memento uses 'pred_answer' (FRAMES/MuSiQue) and 'action_f1' (Mind2Web).

memento_mw_results = load_existing_results(MEMENTO_MIND2WEB_RESULTS)
memento_fr_results = load_jsonl(MEMENTO_FRAMES_RESULTS)
memento_mq_results = load_jsonl(MEMENTO_MUSIQUE_RESULTS)

memento_mind2web_score = score_mind2web(memento_mw_results)
# Memento FRAMES uses 'pred_answer' and 'ground_truth'
memento_frames_score   = score_contains_match(memento_fr_results, "pred_answer", "ground_truth")
# Memento MuSiQue uses 'pred_answer' and 'answer'
memento_musique_score  = score_contains_match(memento_mq_results, "pred_answer", "answer")

print("=" * 50)
print("Memento Results")
print("=" * 50)
print(f"Mind2Web  Avg Action F1:  {memento_mind2web_score:.1f}%  ({len(memento_mw_results)} questions)")
print(f"FRAMES    Accuracy:       {memento_frames_score:.1f}%  ({len(memento_fr_results)} questions)")
print(f"MuSiQue   Contains-Match: {memento_musique_score:.1f}%  ({len(memento_mq_results)} questions)")

---
## 4 — Summary

Aggregate results for AWM and Memento across all three agentic task datasets.

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────

summary = pd.DataFrame([
    {
        "System":           "AWM",
        "Mind2Web F1 %":    round(awm_mind2web_score, 1),
        "FRAMES Acc %":     round(awm_frames_score,   1),
        "MuSiQue CM %":     round(awm_musique_score,  1),
        "Avg %":            round(
            (awm_mind2web_score + awm_frames_score + awm_musique_score) / 3, 1
        ),
    },
    {
        "System":           "Memento",
        "Mind2Web F1 %":    round(memento_mind2web_score, 1),
        "FRAMES Acc %":     round(memento_frames_score,   1),
        "MuSiQue CM %":     round(memento_musique_score,  1),
        "Avg %":            round(
            (memento_mind2web_score + memento_frames_score + memento_musique_score) / 3, 1
        ),
    },
])

summary = summary.set_index("System")
print(summary.to_markdown())
summary

In [ ]:
# ── Reference: Published 250-question Results ─────────────────────────────────
# The numbers below come from the completed evaluation runs.
# Use these to verify that your reproduced results match.

reference = pd.DataFrame([
    {
        "System":        "AWM",
        "Mind2Web F1 %": 50.2,
        "FRAMES Acc %":  20.4,
        "MuSiQue CM %":  24.0,
        "Avg %":         round((50.2 + 20.4 + 24.0) / 3, 1),
        "Notes":         "DuckDuckGo web search; text-embedding-ada-002 retrieval",
    },
    {
        "System":        "Memento",
        "Mind2Web F1 %": 43.8,
        "FRAMES Acc %":  16.0,
        "MuSiQue CM %":  14.8,
        "Avg %":         round((43.8 + 16.0 + 14.8) / 3, 1),
        "Notes":         "CBR sup-simcse-bert retrieval + DuckDuckGo",
    },
]).set_index("System")

print("Reference Results (250-question runs):")
print(reference.to_markdown())
reference

---
## Notes & Observations

### AWM vs Memento

- **AWM outperforms Memento** on all three datasets, with an average score of ~31.5% vs ~24.9%.
- The largest gap is on **MuSiQue** (24.0% vs 14.8%), suggesting AWM's workflow memory is more effective for compositional multi-hop reasoning.
- **Mind2Web F1** is the strongest signal: AWM's higher F1 (50.2% vs 43.8%) indicates it predicts action type and target elements more accurately, likely because text-embedding-ada-002 retrieves more relevant past web navigation workflows.

### Metric Notes

- **Mind2Web Action F1** provides partial credit: a prediction that gets the action type right but the wrong element scores > 0. Success rate (exact match) is 0% for both systems, consistent with the difficulty of exact web action reproduction.
- **Contains-match** for FRAMES and MuSiQue may slightly overestimate accuracy for very short gold answers (e.g., single-word answers like "yes" or "no" that appear coincidentally in longer responses). Results should be interpreted with this in mind.

### Reproducibility

- AWM requires an OpenAI API key (text-embedding-ada-002 + GPT-4 calls).
- Memento requires a Python 3.11 venv and will fail with Python 3.10 or 3.13 due to MCP server dependency constraints.
- DuckDuckGo search results are non-deterministic; re-runs may produce slightly different scores.